In [ ]:
import os
import time
import random
import pandas as pd
from nba_api.stats.endpoints import (
    LeagueDashTeamStats,
    LeagueGameFinder,
    LeagueDashPlayerStats,
 )

output_dir = "/Users/donma/Code/NBA-Analytics/data"
os.makedirs(output_dir, exist_ok=True)

seasons = [f"{y}-{str(y + 1)[-2:]}" for y in range(2010, 2026)]
seasons[:3], seasons[-1]

def fetch_df(endpoint_cls, **kwargs):
    last_error = None
    for attempt in range(4):
        try:
            endpoint = endpoint_cls(timeout=60, **kwargs)
            return endpoint.get_data_frames()[0]
        except Exception as exc:
            last_error = exc
            wait = 2 + (attempt * 4) + random.uniform(0.5, 1.5)
            print(f"retry {attempt + 1}/4 for {endpoint_cls.__name__}: {exc} (sleep {wait:.1f}s)")
            time.sleep(wait)
    raise last_error

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo


pacific = datetime.strptime("2026-05-31", "%Y-%m-%d").astimezone(ZoneInfo("America/Los_Angeles")) # Adjust this date as needed for testing

today_str = pacific.strftime('%m/%d/%Y')
filename_today = pacific.strftime('%Y_%m_%d.csv')
    
    # Determine the current NBA season based on the date
if pacific.month < 10:
        season = f"{pacific.year - 1}-{str(pacific.year)[2:]}"
else:
        season = f"{pacific.year}-{str(pacific.year + 1)[2:]}"

season_types = ['Regular Season', 'Playoffs']
final_games_df = None

print(f"Downloading today's data for {today_str} (season {season})")
for season_type in season_types:
        print(f"Fetching data for {season_type}...")
        games_df = fetch_df(
            LeagueGameFinder,
            date_from_nullable=today_str,
            date_to_nullable=today_str,
            season_type_nullable=season_type,
            league_id_nullable='00',
        )
        games_df['season'] = season
        games_df['season_type'] = season_type
        games_df['ingestion_date'] = pacific.strftime('%Y-%m-%d')
        final_games_df = games_df if final_games_df is None else pd.concat(
            [final_games_df, games_df],
            ignore_index=True,
        )


final_games_df.head()


In [ ]:
import os
import re
import pandas as pd
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
output_dir = "/Users/donma/Code/NBA-Analytics/today_data"
# Function to get the last run date from the CSV files in output_dir
def get_last_run_date(directory):
    latest_date = None
    pattern = re.compile(r'\d{4}_\d{2}_\d{2}(?=\.csv$)')
    
    if os.path.exists(directory):
        for filename in os.listdir(directory):
            match = pattern.search(filename)
            if match:
                date_str = match.group()
                try:
                    file_date = datetime.strptime(date_str, "%Y_%m_%d").date()
                    if latest_date is None or file_date > latest_date:
                        latest_date = file_date
                except ValueError:
                    continue
    return latest_date

## use Pacific time to determine the date range
pacific = datetime.now().astimezone(ZoneInfo("America/Los_Angeles"))
today_date = pacific.date()

last_run = get_last_run_date(output_dir)

# If we have a last run date, start from the next day, otherwise just use today
if last_run and last_run < today_date:
    start_date = last_run + timedelta(days=1)
else:
    start_date = today_date
start_date = datetime.strptime("2025-10-02", "%Y-%m-%d").date()
today_date = datetime.strptime("2025-10-02", "%Y-%m-%d").date() # NBA season typically starts in October, adjust as needed
start_day_string = start_date.strftime("%m/%d/%Y")
end_day_string = today_date.strftime("%m/%d/%Y")

filename_today = today_date.strftime("%Y_%m_%d.csv")

if pacific.month < 10:  # NBA seasons typically begin in October
    season_str = f"{pacific.year - 1}-{str(pacific.year)[2:]}"
else:
    season_str = f"{pacific.year}-{str(pacific.year + 1)[2:]}"

print(f"Downloading delta data from {start_day_string} to {end_day_string} (season {season_str})")

season_types = ["Regular Season", "Playoffs"]

final_teams_df = pd.DataFrame()
final_player_stats_df = pd.DataFrame()
final_games_df = pd.DataFrame()

for st in season_types:
    print(f"Fetching data for {st}...")
    try:
        teams_df = fetch_df(
                LeagueDashTeamStats,
                season=season_str,
                season_type_all_star=st,
                last_n_games=0,
                measure_type_detailed_defense="Base",
                month=0,
                opponent_team_id=0,
                pace_adjust="N",
                per_mode_detailed="Totals",
                period=0,
                plus_minus="Y",
                rank="N",
                # date_from_nullable=start_day_string,
                # date_to_nullable=end_day_string,
            )
        teams_df["season_type"] = st
        final_teams_df = pd.concat([final_teams_df, teams_df], ignore_index=True)
    except Exception as e:
        print(f"Failed to fetch {st} team stats: {e}")

    try:
        player_stats_df = fetch_df(
                LeagueDashPlayerStats,
                season=season_str,
                season_type_all_star=st,
                last_n_games=0,
                measure_type_detailed_defense="Base",
                month=0,
                opponent_team_id=0,
                pace_adjust="N",
                per_mode_detailed="Totals",  # Often 'Totals' is preferred for delta
                period=0,
                plus_minus="Y",
                rank="N",
                date_from_nullable=start_day_string,
                date_to_nullable=end_day_string,
            )
        player_stats_df["season_type"] = st
        final_player_stats_df = pd.concat([final_player_stats_df, player_stats_df], ignore_index=True)
    except Exception as e:
        print(f"Failed to fetch {st} player stats: {e}")

try:
    games_df = fetch_df(
        LeagueGameFinder,
        date_from_nullable=start_day_string,
        date_to_nullable=end_day_string,
        league_id_nullable="00",
    )
    final_games_df = games_df
    
except Exception as e:
    print(f"Failed to fetch games: {e}")

print("Data fetching complete. Sample data:")
print("Teams DataFrame:")
print(final_teams_df.head())
print("\nPlayer Stats DataFrame:")
print(final_player_stats_df.head())
print("\nGames DataFrame:")
print(final_games_df.head())

In [ ]:


def fetch_df(endpoint_cls, **kwargs):
    last_error = None
    for attempt in range(3):
        try:
            endpoint = endpoint_cls(timeout=60, **kwargs)
            return endpoint.get_data_frames()[0]
        except Exception as exc:
            last_error = exc
            wait = 2 + (attempt * 3) + random.uniform(0.5, 1.5)
            print(f"retry {attempt + 1}/3 for {endpoint_cls.__name__}: {exc} (sleep {wait:.1f}s)")
            time.sleep(wait)
    raise last_error


def fetch_todays_games_to_s3(**kwargs):
    output_dir = "/tmp/nba_daily_data"
    os.makedirs(output_dir, exist_ok=True)

    pacific = datetime.now(ZoneInfo('America/Los_Angeles'))
    pacific = datetime.strptime("2026-05-29", "%Y-%m-%d").astimezone(ZoneInfo("America/Los_Angeles")) # Adjust this date as needed for testing
    today_str = pacific.strftime('%m/%d/%Y')
    
    
    # Determine the current NBA season based on the date
    if pacific.month < 10:
        season = f"{pacific.year - 1}-{str(pacific.year)[2:]}"
    else:
        season = f"{pacific.year}-{str(pacific.year + 1)[2:]}"

    season_types = ['Regular Season', 'Playoffs']
    final_games_df = None
    filename_today ='games_'+season +'_'+ "2026_05_29.csv"
    print(f"Downloading today's data for {today_str} (season {season})")
    for season_type in season_types:
        print(f"Fetching data for {season_type}...")
        games_df = fetch_df(
            LeagueGameFinder,
            date_from_nullable=today_str,
            date_to_nullable=today_str,
            season_type_nullable=season_type,
            league_id_nullable='00',
        )
        games_df['season'] = season
        games_df['season_type'] = season_type
        games_df['ingestion_date'] = pacific.strftime('%Y-%m-%d')
        final_games_df = games_df if final_games_df is None else pd.concat(
            [final_games_df, games_df],
            ignore_index=True,
        )
    return final_games_df, filename_today
final_games_df, filename_today = fetch_todays_games_to_s3()
final_games_df.to_csv(os.path.join(output_dir, filename_today), index=False)

In [ ]:
pd.set_option("display.max_columns", None)
print("Player DataFrame:")
print(player_stats_df[player_stats_df['PLAYER_NAME'] == 'LeBron James'])

In [ ]:
from datetime import date


season_types = ["Regular Season", "Playoffs"]

for season in seasons:
    final_teams_df = None
    final_games_df = None
    final_player_stats_df = None
    for season_type in season_types:
        print(f"Downloading season {season} ({season_type})...")
        teams_df = fetch_df(
            LeagueDashTeamStats,
            season=season,
            season_type_all_star=season_type,
            last_n_games=0,
            measure_type_detailed_defense="Base",
            month=0,
            opponent_team_id=0,
            pace_adjust="N",
            per_mode_detailed="Totals",
            period=0,
            plus_minus="Y",
            rank="N",
        )
        player_stats_df = fetch_df(
            LeagueDashPlayerStats,
            season=season,
            season_type_all_star=season_type,
            last_n_games=0,
            measure_type_detailed_defense="Base",
            month=0,
            opponent_team_id=0,
            pace_adjust="N",
            per_mode_detailed="PerGame",
            period=0,
            plus_minus="Y",
            rank="N",
          
        )
        games_df = fetch_df(
            LeagueGameFinder,
            season_nullable=season,
            season_type_nullable=season_type,
            league_id_nullable="00",
        )

        teams_df["season"] = season
        teams_df["season_type"] = season_type
        teams_df["ingestion_date"] = datetime.now().strftime("%Y-%m-%d")
        player_stats_df["season"] = season
        player_stats_df["season_type"] = season_type
        player_stats_df["ingestion_date"] = datetime.now().strftime("%Y-%m-%d")
        games_df["season"] = season
        games_df["season_type"] = season_type
        games_df["ingestion_date"] = datetime.now().strftime("%Y-%m-%d")
        season_type_slug = season_type.lower().replace(" ", "_")
        if final_teams_df is None:
            final_teams_df = teams_df
        else:
            final_teams_df = pd.concat([final_teams_df, teams_df], ignore_index=True)

        if final_player_stats_df is None:
            final_player_stats_df = player_stats_df
        else:
            final_player_stats_df = pd.concat([final_player_stats_df, player_stats_df], ignore_index=True)
        if final_games_df is None:
            final_games_df = games_df
        else:
            final_games_df = pd.concat([final_games_df, games_df], ignore_index=True)
    filename_today = datetime.today().strftime("%Y_%m_%d.csv")
    teams_path = os.path.join(output_dir, f"teams_{season}_{filename_today}")
    player_stats_path = os.path.join(output_dir, f"player_stats_{season}_{filename_today}")
    games_path = os.path.join(output_dir, f"games_{season}_{filename_today}")

    final_teams_df.to_csv(teams_path, index=False)
    final_player_stats_df.to_csv(player_stats_path, index=False)
    final_games_df.to_csv(games_path, index=False)


In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import os
import random
import pandas as pd
from nba_api.stats.endpoints import (
    LeagueDashTeamStats,
    LeagueGameFinder,
    LeagueDashPlayerStats,
 )

output_dir = "/Users/donma/Code/NBA-Analytics/today_data"
os.makedirs(output_dir, exist_ok=True)

def fetch_df(endpoint_cls, **kwargs):
    last_error = None
    for attempt in range(4):
        try:
            endpoint = endpoint_cls(timeout=60, **kwargs)
            return endpoint.get_data_frames()[0]
        except Exception as exc:
            last_error = exc
            wait = 2 + (attempt * 4) + random.uniform(0.5, 1.5)
            print(f"retry {attempt + 1}/4 for {endpoint_cls.__name__}: {exc} (sleep {wait:.1f}s)")
            time.sleep(wait)
    raise last_error

pacific = datetime.now().astimezone(ZoneInfo("America/Los_Angeles"))

today_str = pacific.strftime("%m/%d/%Y")
filename_today = pacific.strftime("%Y_%m_%d.csv")

if pacific.month < 10:  # NBA seasons typically begin in October
    season = f"{pacific.year - 1}-{str(pacific.year)[2:]}"
else:
    season = f"{pacific.year}-{str(pacific.year + 1)[2:]}"

season_type = ['Regular Season', 'Playoffs']
final_games_df = None
print(f"Downloading today's data for {today_str} (season {season})")
for st in season_type:
    print(f"Fetching data for {st}...")
    games_df = fetch_df(
        LeagueGameFinder,
        date_from_nullable=today_str,
        date_to_nullable=today_str,
        season_type_nullable=st,
        league_id_nullable="00"
    )
    games_df["season"] = season
    games_df["season_type"] = st
    final_games_df = games_df if final_games_df is None else pd.concat([final_games_df, games_df], ignore_index=True)

games_path = os.path.join(output_dir, f"games_{season}_{filename_today}")

final_games_df.to_csv(games_path, index=False)
final_games_df.head()

In [ ]:
from nba_api.stats.endpoints import playercareerstats

# Nikola Jokić
career = playercareerstats.PlayerCareerStats(player_id='203999')

# pandas data frames (optional: pip install pandas)
career.season_totals_regular_season.get_data_frame()

# json
career.get_json()

# dictionary
career.get_dict()

In [ ]:
from nba_api.stats.endpoints import leaguegamefinder, leaguedashteamstats, playergamelog, boxscoretraditionalv2
from nba_api.stats.static import teams, players
import pandas as pd
from datetime import datetime, timedelta
import boto3

s3 = boto3.client('s3')
bucket = 'your-nba-bucket'

def get_all_teams():
    return teams.get_teams()

def ingest_games(season='2024-25', season_type='Regular Season'):
    game_finder = leaguegamefinder.LeagueGameFinder(season_nullable=season, season_type_nullable=season_type)
    df = game_finder.get_data_frames()[0]
    date_str = datetime.now().strftime('%Y-%m-%d')
    # df.to_csv(f'games_{date_str}.csv', index=False)
    # s3.upload_file(f'games_{date_str}.csv', bucket, f'raw/games/{date_str}/games.csv')
    return df

# Similar functions for team stats, player game logs, box scores (per GAME_ID)
# For box scores (detailed with current score, FG):
def get_box_score(game_id):
    box = boxscoretraditionalv2.BoxScoreTraditionalV2(game_id=game_id)
    team_stats = box.team_stats.get_data_frame()  # TEAM_ID, PTS, FGM, FGA, etc.
    player_stats = box.player_stats.get_data_frame()
    return team_stats, player_stats

print("Fetching today's data...")


In [ ]:
# Example: Fetching NBA players using nba_api (optional, not Spark)
from nba_api.stats.static import players
nba_players = players.get_active_players()
print(f"Number of players fetched: {len(nba_players)}")
nba_players[:5]

In [ ]:
import awswrangler as wr
import boto3
from pyspark.sql import SparkSession
from delta import *
builder = (
    SparkSession.builder.master("local[4]")
    .appName("parallel")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
session = boto3.Session()

games_path = "s3://nbaanalysisproject/silver/Games/"

df = spark.read.format("delta").load(games_path)
print(df.head())


In [ ]:
# Get unique team names
unique_teams = df.select('team').distinct().rdd.flatMap(lambda x: x).collect()
print(unique_teams)
# Get all players from team 2TM
team_2tm_players = df.filter(col('team') == '2TM')
team_2tm_players.show()

In [ ]:
df.columns

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_extract, lit, array_contains
import os

s3_players_path = "s3://bucketname/gold/playerdata/season=2016/team=BOS/part.snappy.parquet"

# Read the Parquet file into a Spark DataFrame
# (Assumes AWS credentials are set in environment or Hadoop config)
df = pd.read.parquet(s3_players_path)

# ---- User parameters ----
season = 2025  # Set your season
season_df = df.filter(col('season') == season)


In [ ]:
# ---- Identify Team Change Players ----
team_change_df = season_df.filter(col('team').isin(['2TM', '3TM', '4TM']))
normal_df = season_df.filter(~col('team').isin(['2TM', '3TM', '4TM']))

# ---- Extract Number of Teams ----
team_change_df = team_change_df.withColumn('team_count', regexp_extract(col('team'), r'(\d)', 1).cast('int'))
normal_df = normal_df.withColumn('team_count', lit(1))

# ---- Mark players with multiple teams ----
multi_team_player_ids = [row['player_id'] for row in team_change_df.select('player_id').distinct().collect()]
from pyspark.sql.functions import when
normal_df = normal_df.withColumn('changed_team', when(col('player_id').isin(multi_team_player_ids), 1).otherwise(0))

# ---- Drop 2TM, 3TM, 4TM rows ----
final_gold_df = normal_df

# ---- Filter Only Valid Teams ----
valid_teams = [
    'SAC','HOU','MIA','TOR','MEM','ATL','NOP','PHO','CLE','UTA','MIL',
    'NYK','POR','LAL','WAS','CHO','ORL','PHI','SAS','OKC','LAC','MIN',
    'BOS','IND','DEN','GSW','CHI','DAL','BRK','DET'
]
final_gold_df = final_gold_df.filter(col('team').isin(valid_teams + ['MULTI']))
#check final gold_df rows count > 0
final_gold_df.count()

In [ ]:
final_gold_df.filter(col('player_id') == 'anderky01').show()

In [ ]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from pyspark.sql.functions import col, regexp_extract, lit, when
from awsglue.context import GlueContext
from awsglue.job import Job
from awsgluedq.transforms import EvaluateDataQuality
import re


args = getResolvedOptions(sys.argv, ['JOB_NAME','season','silver_path','gold_path'])

season = args["season"]
silver_path = args["silver_path"]
gold_path = args["gold_path"]


sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
job.init(args['JOB_NAME'], args)


df = spark.read.parquet(silver_path)

from pyspark.sql.functions import col

# Assuming `df` is your Spark DataFrame
df = df.withColumn("w", col("w").cast("integer")) \
       .withColumn("n_rtg", col("n_rtg").cast("double")) \
       .withColumn("pace", col("pace").cast("double")) \
       .withColumn("playoffs", col("playoffs").cast("boolean")) \
       .withColumn("orb_percent", col("orb_percent").cast("double")) \
       .withColumn("f_tr", col("f_tr").cast("double")) \
       .withColumn("drb_percent", col("drb_percent").cast("double")) \
       .withColumn("opp_tov_percent", col("opp_tov_percent").cast("double")) \
       .withColumn("abbreviation", col("abbreviation").cast("string")) \
       .withColumn("ft_fga", col("ft_fga").cast("double")) \
       .withColumn("tov_percent", col("tov_percent").cast("double")) \
       .withColumn("o_rtg", col("o_rtg").cast("double")) \
       .withColumn("d_rtg", col("d_rtg").cast("double")) \
       .withColumn("age", col("age").cast("double")) \
       .withColumn("ts_percent", col("ts_percent").cast("double")) \
       .withColumn("arena", col("arena").cast("string")) \
       .withColumn("opp_e_fg_percent", col("opp_e_fg_percent").cast("double")) \
       .withColumn("team", col("team").cast("string")) \
       .withColumn("l", col("l").cast("integer")) \
       .withColumn("x3p_ar", col("x3p_ar").cast("double")) \
       .withColumn("e_fg_percent", col("e_fg_percent").cast("double")) \
       .withColumn("opp_ft_fga", col("opp_ft_fga").cast("double")) \
       .withColumn("ast_per_game", col("ast_per_game").cast("double")) \
       .withColumn("x3p_per_game", col("x3p_per_game").cast("double")) \
       .withColumn("orb_per_game", col("orb_per_game").cast("double")) \
       .withColumn("fta_per_game", col("fta_per_game").cast("double")) \
       .withColumn("trb_per_game", col("trb_per_game").cast("double")) \
       .withColumn("fg_percent", col("fg_percent").cast("double")) \
       .withColumn("x3pa_per_game", col("x3pa_per_game").cast("double")) \
       .withColumn("stl_per_game", col("stl_per_game").cast("double")) \
       .withColumn("ft_percent", col("ft_percent").cast("double")) \
       .withColumn("drb_per_game", col("drb_per_game").cast("double")) \
       .withColumn("mp_per_game", col("mp_per_game").cast("double")) \
       .withColumn("g", col("g").cast("integer")) \
       .withColumn("x2pa_per_game", col("x2pa_per_game").cast("double")) \
       .withColumn("x3p_percent", col("x3p_percent").cast("double")) \
       .withColumn("pts_per_game", col("pts_per_game").cast("double")) \
       .withColumn("ft_per_game", col("ft_per_game").cast("double")) \
       .withColumn("tov_per_game", col("tov_per_game").cast("double")) \
       .withColumn("pf_per_game", col("pf_per_game").cast("double")) \
       .withColumn("fg_per_game", col("fg_per_game").cast("double")) \
       .withColumn("blk_per_game", col("blk_per_game").cast("double")) \
       .withColumn("x2p_percent", col("x2p_percent").cast("double")) \
       .withColumn("fga_per_game", col("fga_per_game").cast("double")) \
       .withColumn("x2p_per_game", col("x2p_per_game").cast("double"))


final_gold_df = df[df['season']==season]





row_count = final_gold_df.count()
if row_count <= 0:
    raise ValueError("Sanity check failed: final_gold_df is empty!")
else:
    print(f"Sanity check passed: final_gold_df has {row_count} rows.")

final_gold_df.write.mode("overwrite").partitionBy("season").parquet(gold_path)

job.commit()

In [ ]:
import pyspark
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder.appName("NBA Analytics").getOrCreate()
games_path = "s3://bucketname/silver/games"

df = spark.read.format("delta").load("games_path")
df.show()

In [ ]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from awsgluedq.transforms import EvaluateDataQuality

args = getResolvedOptions(sys.argv, ['JOB_NAME'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)

# Default ruleset used by all target nodes with data quality enabled
DEFAULT_DATA_QUALITY_RULESET = """
    Rules = [
        ColumnCount > 0
    ]
"""

Games_df = spark.read('delta_lake')


# Script generated for node Amazon S3
AmazonS3_node1779983780639 = glueContext.create_dynamic_frame.from_options(format_options={"quoteChar": "\"", "withHeader": True, "separator": ",", "optimizePerformance": False}, connection_type="s3", format="csv", connection_options={"paths": ["s3://nbaanalysisproject/bronze/raw/Games/"], "recurse": True}, transformation_ctx="AmazonS3_node1779983780639")

# Script generated for node Amazon S3
EvaluateDataQuality().process_rows(frame=AmazonS3_node1779983780639, ruleset=DEFAULT_DATA_QUALITY_RULESET, publishing_options={"dataQualityEvaluationContext": "EvaluateDataQuality_node1779983749665", "enableDataQualityResultsPublishing": True}, additional_options={"dataQualityResultsPublishing.strategy": "BEST_EFFORT", "observations.scope": "ALL"})
additional_options = {"path": "s3://nbaanalysisproject/silver/Games/", "write.parquet.compression-codec": "snappy"}
AmazonS3_node1779983806756_df = AmazonS3_node1779983780639.toDF()
AmazonS3_node1779983806756_df.write.format("delta").options(**additional_options).partitionBy("ingestion_date").mode("append").save()

job.commit()

In [ ]:
from nba_api.stats.endpoints import playbyplayv3, boxscoreadvancedv3
import pandas as pd
from datetime import datetime
import time

from datetime import datetime
import time

def parse_clock_to_seconds(clock_str: str, period: int) -> int:
    """Convert clock 'MM:SS' to total seconds remaining in the entire game."""
    if not isinstance(clock_str, str) or ':' not in clock_str:
        return 0
    try:
        mins, secs = map(int, clock_str.split(':'))
        seconds_in_current_period = mins * 60 + secs
        # NBA quarters = 12 minutes (except OT)
        remaining_periods = max(0, 4 - period)
        return seconds_in_current_period + (remaining_periods * 12 * 60)
    except:
        return 0
    
def get_play_by_play(game_id: str):
    pbp = playbyplayv3.PlayByPlayV3(game_id=game_id)
    df = pbp.play_by_play.get_data_frame()
    return df

# Example
game_id = "0022001076"   # Format: 00225 = season 2025-26
pbp_df = get_play_by_play(game_id)

In [ ]:
boxscoreadvancedv3.BoxScoreAdvancedV3(game_id=game_id).get_data_frames()[1]


In [ ]:
import pandas as pd


def create_in_game_snapshots(game_id: str):
    """
    Creates ~1-minute interval snapshots from Play-by-Play data.
    """
    pbp = get_play_by_play(game_id)
    pbp['GAME_DATE'] = datetime.now().date()  # or get from schedule
    pbp['GAME_ID'] = game_id


    if pbp.empty:
        print(f"No data returned for game {game_id}")
        return pd.DataFrame()
    
    # Use your exact column names (case-sensitive)
    required_cols = ['actionNumber', 'clock', 'period', 'scoreHome', 'scoreAway']
    for col in required_cols:
        if col not in pbp.columns:
            print(f"Warning: Column '{col}' not found. Available: {list(pbp.columns)}")
    
    # Sort chronologically
    pbp = pbp.sort_values('actionNumber').reset_index(drop=True)
    
    snapshots = []
    
    # Get final result from last row
    final_row = pbp.iloc[-1]
    final_home = int(final_row.get('scoreHome', 0)) if pd.notna(final_row.get('scoreHome')) else 0
    final_away = int(final_row.get('scoreAway', 0)) if pd.notna(final_row.get('scoreAway')) else 0
    final_win_home = 1 if final_home > final_away else 0
    
    last_snapshot_time = float('inf')   # Total seconds remaining
    snapshot_interval = 60              # 1 minute
    
    for i in range(len(pbp)):
        row = pbp.iloc[i]
        
        time_remaining = parse_clock_to_seconds(row['clock'], int(row['period']))
        current_score_diff = int(row.get('scoreHome', 0)) - int(row.get('scoreAway', 0))
        
        # Create snapshot every ~60 seconds + first and last events
        if (last_snapshot_time - time_remaining >= snapshot_interval) or \
           i == 0 or i == len(pbp) - 1:
            
            snapshot = {
                'GAME_ID': game_id,
                'ACTION_NUMBER': int(row['actionNumber']),
                'PERIOD': int(row['period']),
                'TIME_REMAINING': time_remaining,           # Seconds left in game
                'CURRENT_SCORE_DIFF': current_score_diff,   # Home - Away
                'CURRENT_FG_DIFF': 0,                       # Placeholder - can be enhanced
                'HOME': 1,                                  # Modeling home team win probability
                # === Rolling / Pre-game features (you will merge these later) ===
                'ROLL_WIN_PCT_10': None,
                'ROLL_FG_PCT_10': None,
                'ROLL_EFG_PCT_10': None,
                'OPP_ROLL_WIN_PCT_10': None,
                'REST_DAYS': None,
                'BACK_TO_BACK': None,
                # Target
                'FINAL_WIN_HOME': final_win_home
            }
            snapshots.append(snapshot)
            last_snapshot_time = time_remaining

    df_snapshots = pd.DataFrame(snapshots)
    print(f"Game {game_id}: Created {len(df_snapshots)} snapshots (~1 per minute)")
    
    return df_snapshots

In [ ]:
create_in_game_snapshots("0022001076")